# Pauli Exponential

- The Pauli exponential gadget is one of the fundamental building blocks in quantum algorithms, used to implement rotations around axes defined by Pauli operators.

    $$e^{-i \frac{\theta}{2} P}$$

    where $ P $ is a Pauli operator (e.g., $ X_0 Y_1 Z_2 $) and $ \theta $ is the rotation angle.

- It is the key primitive for the Trotterization of Hamiltonian evolution:

    $$e^{-i H t} \approx \left( \prod_{j} e^{-i \frac{\theta_j}{2} P_j} \right)^n$$

    where $ H = \sum_j \theta_j P_j $ is the Hamiltonian expressed as a sum of Pauli operators.

- In guppy the pauli exponential is implemented using a python meta function which builds the guppy functions based on:

  - The `zixy` Pauli string $ X_0 Y_1 Z_2 \cdots Z_N $ to be exponentiated
  - The number of state qubits the Pauli string acts on
  - The CX ladder method (linear or logarithmic depth)
  - The RZ decomposition method

## `zixy` Objects

- `zqp.String` represents a single Pauli string on a fixed number of qubits.

  Example:
  ```python
  pauli_string = zqp.String.from_str("Z0 X1 Y2", 3)
  ```

- `zqp.Strings` represents a collection of Pauli strings, which is useful for primitives like Pauli select.

  Example:
  ```python
  pauli_strings = zqp.Strings.from_str("X0 X1, Y1 Z2, Z0", 3)
  ```

- `zqp.RealTerm` represents one weighted Pauli term, combining a real coefficient with a Pauli string.

  Example:
  ```python
  real_term = next(iter(zqp.RealTermSum.from_str("(0.5, Z0 X1)").to_terms()))
  ```

- `zqp.RealTermSum` represents a sum of weighted Pauli terms. This is the natural object for Hamiltonians, and it is also where helpers like `to_sparse_matrix()` live.

  Example:
  ```python
  hamiltonian = zqp.RealTermSum.from_str("(0.5, Z0 X1), (-0.2, Y0 Y1)")
  ```

In this notebook we use `zqp` for string-level `zixy` objects and `zqp` for term and term-sum objects. We use a `String` to describe the Pauli operator we want to exponentiate, and a `RealTermSum` when we want a matrix representation for comparison with `scipy.linalg.expm`.

In [ ]:
from guppyalgos.primitives.pauli.pauli_exp import pauli_exp
from guppylang import guppy
from guppylang.std.builtins import comptime, array
from guppyalgos.tests.helpers import get_unitary
from scipy.linalg import expm
import numpy as np
from guppyalgos.tests.helpers import assert_allclose_ignorephase
import zixy.qubit.pauli as zqp
import zixy.qubit.pauli as zqp
from guppyalgos.primitives.subroutines.ladders import CXLadderLinear
from guppylang.std.quantum import rz, qubit
from guppylang.std.angles import angle

In [ ]:
n_state_qubits = 3
pauli_string = zqp.String.from_str("Z0 X1 Y2", n_state_qubits)

cx_ladder_method = CXLadderLinear

rz_method = rz

pauli_gadget = pauli_exp(pauli_string, n_state_qubits, cx_ladder_method, rz_method)

- The defaults are CXLadderLog and rz

- This can then be used within a main function, where the angle is specified as a compile time parameter.

In [ ]:
theta = 0.7

@guppy
def main(state_qreg:array[qubit,n_state_qubits]) -> None:
    pauli_gadget(state_qreg, angle(theta))

- We can compare the unitary generated by our pauli exponential implementation against the expected unitary using scipy's expm function up to a phase

In [ ]:
guppy_u = get_unitary(main, n_state_qubits)

In [ ]:
pauli_mat = zqp.RealTermSum.from_str(str(pauli_string), n_state_qubits).to_sparse_matrix(True).todense()
u_mat = expm(-1j * (0.5* np.pi *(theta)) * pauli_mat)

In [ ]:
assert_allclose_ignorephase(u_mat, guppy_u)

## RUS RZ Example

- The repeat-until-success RZ rotation is a method for implementing arbitrary single-qubit rotations around the Z-axis using a probabilistic approach. This technique leverages resource states and measurements to achieve the desired rotation with high fidelity.
- Curerrently as we do not have resource states we are just using a dummy resource state for demonstration purposes.

In [8]:
from guppyalgos.primitives.rotations import repeat_until_success_rz, dummy_theta_resource_state
from guppyalgos.utils import qarray, transversal
from guppylang.std.debug import state_output
from guppylang.std.quantum import discard_array, h
from selene_sim import QuantumReplay, Quest
rus_rz = repeat_until_success_rz(dummy_theta_resource_state)

pauli_gadget = pauli_exp(pauli_string, n_state_qubits, cx_ladder_method, rus_rz)

theta = 0.7

@guppy
def main() -> None:
    state_qreg = qarray(n_state_qubits)
    transversal(h, state_qreg)
    pauli_gadget(state_qreg, angle(theta))
    state_output("result_state", state_qreg)
    discard_array(state_qreg)

n_repeats = 3
# fail 3 times and then succeed
desired_rus_measurements = [[False] * n_repeats + [True]]

rus_replay_sim = QuantumReplay(simulator=Quest(), measurements=desired_rus_measurements)
em_result = (
    main.emulator(n_state_qubits+1).with_simulator(rus_replay_sim).with_shots(1).run()
)
# assert that the correct angle is found for every number of failures
for shot_result in em_result.results:
    states = Quest.extract_states_dict(shot_result)
    rus_sv = states["result_state"].state
    rus_sv = rus_sv[2**(n_state_qubits):] #discard the |0> state of the resource qubit


had_state = ((1/(np.sqrt(2)))**n_state_qubits) * np.ones(2**n_state_qubits)

pauli_mat = zqp.RealTermSum.from_str(str(pauli_string), n_state_qubits).to_sparse_matrix(True).todense()

u_mat = expm(1j * (-0.5* np.pi *(theta)) * pauli_mat) #note sign change

sv = u_mat @ had_state

assert_allclose_ignorephase(sv, rus_sv)
